# LG-03: Human-in-the-loop 人工介入

> **阶段**: LG-03 | **难度**: 中级 | **预计时长**: 3-4 小时

## 学习目标

- 建立 Human-in-the-loop 在图执行中的位置
- 区分 `interrupt()`、`Command(resume=...)`、`checkpointer` 各自负责的事情
- 跑通一次“暂停 → 查看 state → 人工决策 → 恢复执行”的完整链路
- 观察低风险自动通过、高风险暂停审批、拒绝发布、编辑后发布四种结果
- 理解 `interrupt_before` 这种声明式中断适合拦截写操作


In [ ]:
# 安装依赖（如未安装请取消注释）
# !pip install -U langgraph langchain-core

## 1. 核心概念

Human-in-the-loop（HITL）指的是：图执行到关键节点时主动暂停，把当前任务和状态交给人类判断，再根据人工输入继续执行。

在 LangGraph 里，HITL 不是普通的 `input()`，也不是抛异常。它由三个部分组成：

1. `interrupt(payload)`：在节点内部暂停，并把需要人工处理的信息交出去。
2. `checkpointer`：保存暂停时的 state，让后续可以从同一个位置继续。
3. `Command(resume=value)`：把人工决策传回图中，恢复被暂停的节点。

本节要观察的证据：图是否真的停住、停住时 state 是否还在、人工输入是否能改变后续发布结果。


## 2. 概念边界

HITL 解决的是“关键步骤需要人类确认”的问题。它不负责替代权限系统、审核规则库或前端表单系统。

常见边界：

- 异常处理：异常表示流程失败；`interrupt()` 表示流程暂停等待输入。
- 普通分支：条件边可以自动选择路径；HITL 用在人类必须参与的路径。
- 日志记录：审计日志只记录发生了什么；HITL 会改变流程是否继续。
- 前端确认框：确认框只是界面；HITL 还要保存图状态并支持恢复。


## 3. 案例背景：内容发布审批

一个运营助手会根据活动主题生成营销文案，并准备发布到公众号、短信或 App Push。

用户可能这样问：

- `帮我生成 618 智能空调促销文案，并发布到公众号`
- `这条会员日短信风险不高就直接发`
- `如果文案包含绝对化表达，先交给审核员确认`
- `审核员可以通过、拒绝，也可以修改后通过`

这类系统不能只靠模型自动完成。内容发布是写操作，错误文案会产生真实影响。图需要在高风险节点暂停，把草稿、命中的规则和风险评分交给人类确认。


## 4. 设计思考

运行代码前，先判断这个案例里哪些步骤可以自动做，哪些步骤要停下来等人：

1. 读取发布请求：自动。
2. 根据本地规则计算风险：自动。
3. 高风险内容是否发布：需要人工确认。
4. 低风险内容发布：可以自动继续。
5. 真正发布前的最后一步：可以用声明式中断拦截。

本节会把这个判断落到图里：风险评分决定是否触发 `interrupt()`，人工决策通过 `Command(resume=...)` 回到同一个节点。


## 5. 内容发布审批案例

本节使用内容发布审批流程作为案例：系统先根据审核规则评估风险，低风险内容自动发布，高风险内容暂停并等待人工审批。你会看到 LangGraph 如何保存中断前的状态，并在收到人工决策后从原位置继续执行。


In [ ]:
from __future__ import annotations

from pathlib import Path
from typing import Any, TypedDict
from uuid import uuid4
import json

from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt

DATA_PATH = Path("content_review_data.json")
if not DATA_PATH.exists():
    DATA_PATH = Path("turtorial/LG-03-human-in-the-loop/content_review_data.json")

CONTENT_DATA = json.loads(DATA_PATH.read_text(encoding="utf-8"))

print("内容发布审批案例已就绪")
print("发布请求:", len(CONTENT_DATA["requests"]))
print("审核规则:", len(CONTENT_DATA["policy_rules"]))


## 6. 朴素实现：不暂停，直接发布

下面先写一个最直接的实现：读取草稿、算风险、发布。它看起来省事，但会把高风险文案直接发出去。


In [ ]:
RISK_SCORE_BY_LEVEL = {
    "high": 0.9,
    "medium": 0.55,
    "low": 0.2,
}


def find_request(request_id: str) -> dict[str, Any]:
    return next(item for item in CONTENT_DATA["requests"] if item["request_id"] == request_id)


def assess_risk(draft: str) -> dict[str, Any]:
    matched = []
    score = 0.0
    level = "none"

    for rule in CONTENT_DATA["policy_rules"]:
        terms = [term for term in rule["terms"] if term in draft]
        if not terms:
            continue
        matched.append({
            "rule_id": rule["rule_id"],
            "level": rule["level"],
            "terms": terms,
            "description": rule["description"],
        })
        candidate_score = RISK_SCORE_BY_LEVEL[rule["level"]]
        if candidate_score > score:
            score = candidate_score
            level = rule["level"]

    return {"risk_score": score, "risk_level": level, "matched_rules": matched}


def naive_publish(request_id: str) -> dict[str, Any]:
    request = find_request(request_id)
    risk = assess_risk(request["draft_seed"])
    return {
        "request_id": request_id,
        "publish_status": "published",
        "risk_score": risk["risk_score"],
        "risk_level": risk["risk_level"],
        "matched_rules": risk["matched_rules"],
        "final_content": request["draft_seed"],
    }


naive_result = naive_publish("campaign_high_001")
print("publish_status:", naive_result["publish_status"])
print("risk_score:", naive_result["risk_score"])
print("risk_level:", naive_result["risk_level"])
print("matched_terms:", [term for rule in naive_result["matched_rules"] for term in rule["terms"]])
print("final_content:", naive_result["final_content"])


## 7. 问题观察

上面的输出显示，高风险文案已经变成 `published`。风险评分、命中词和最终内容都说明：这个流程缺少一个“关键动作前暂停”的机制。

HITL 要解决的不是“能不能算出风险”，而是“算出风险之后，图能不能停住，并等待人工决策”。


## 8. 核心实现：用 `interrupt()` 暂停审批

下面定义完整审批图。高风险内容会在 `human_review` 节点暂停；低风险内容会自动通过。


In [ ]:
class ReviewState(TypedDict, total=False):
    request_id: str
    topic: str
    channel: str
    owner_id: str
    draft: str
    risk_score: float
    risk_level: str
    matched_rules: list[dict[str, Any]]
    needs_human_review: bool
    review_decision: str
    reviewer_id: str
    reviewer_comment: str
    revised_draft: str
    publish_status: str
    final_content: str
    audit_log: list[str]


HUMAN_REVIEW_THRESHOLD = 0.75


def append_log(state: ReviewState, message: str) -> list[str]:
    return state.get("audit_log", []) + [message]


def load_request(state: ReviewState) -> dict[str, Any]:
    request = find_request(state["request_id"])
    return {
        "topic": request["topic"],
        "channel": request["channel"],
        "owner_id": request["owner_id"],
        "draft": request["draft_seed"],
        "audit_log": append_log(state, f"读取请求: {request['request_id']}"),
    }


def auto_review(state: ReviewState) -> dict[str, Any]:
    risk = assess_risk(state["draft"])
    needs_human_review = risk["risk_score"] >= HUMAN_REVIEW_THRESHOLD
    return {
        **risk,
        "needs_human_review": needs_human_review,
        "audit_log": append_log(
            state,
            f"自动审核: risk={risk['risk_score']}, needs_human_review={needs_human_review}",
        ),
    }


def human_review(state: ReviewState) -> dict[str, Any]:
    if not state["needs_human_review"]:
        return {
            "review_decision": "auto_approved",
            "reviewer_id": "system",
            "reviewer_comment": "风险低于人工审批阈值，自动通过。",
            "audit_log": append_log(state, "低风险自动通过"),
        }

    decision = interrupt({
        "action": "content_review_required",
        "request_id": state["request_id"],
        "channel": state["channel"],
        "draft": state["draft"],
        "risk_score": state["risk_score"],
        "risk_level": state["risk_level"],
        "matched_rules": state["matched_rules"],
        "options": ["approve", "edit_and_approve", "reject"],
    })

    return {
        "review_decision": decision["decision"],
        "reviewer_id": decision.get("reviewer_id", "reviewer_demo"),
        "reviewer_comment": decision.get("comment", ""),
        "revised_draft": decision.get("revised_draft", ""),
        "audit_log": append_log(state, f"人工审批: {decision['decision']}"),
    }


def publish_content(state: ReviewState) -> dict[str, Any]:
    decision = state["review_decision"]
    if decision in {"approve", "auto_approved"}:
        final_content = state["draft"]
        publish_status = "published"
    elif decision == "edit_and_approve":
        final_content = state["revised_draft"]
        publish_status = "published_after_edit"
    else:
        final_content = ""
        publish_status = "rejected"

    return {
        "final_content": final_content,
        "publish_status": publish_status,
        "audit_log": append_log(state, f"发布结果: {publish_status}"),
    }


builder = StateGraph(ReviewState)
builder.add_node("load_request", load_request)
builder.add_node("auto_review", auto_review)
builder.add_node("human_review", human_review)
builder.add_node("publish_content", publish_content)
builder.add_edge(START, "load_request")
builder.add_edge("load_request", "auto_review")
builder.add_edge("auto_review", "human_review")
builder.add_edge("human_review", "publish_content")
builder.add_edge("publish_content", END)

review_graph = builder.compile(checkpointer=MemorySaver())
print("review_graph compiled")


## 9. 运行链路：暂停、查看 state、恢复执行

先运行一条高风险请求。第一次执行会停在 `human_review`，返回 `__interrupt__`。


In [ ]:
high_config = {"configurable": {"thread_id": "lg03-high-edit"}}
high_first = review_graph.invoke(
    {"request_id": "campaign_high_001"},
    config=high_config,
)
high_snapshot = review_graph.get_state(high_config)
high_interrupt = high_first["__interrupt__"][0]

print("first_result keys:", sorted(high_first.keys()))
print("interrupt action:", high_interrupt.value["action"])
print("interrupt risk_score:", high_interrupt.value["risk_score"])
print("interrupt matched_terms:", [term for rule in high_interrupt.value["matched_rules"] for term in rule["terms"]])
print("snapshot next:", high_snapshot.next)
print("snapshot state keys:", sorted(high_snapshot.values.keys()))


In [ ]:
edit_decision = {
    "decision": "edit_and_approve",
    "reviewer_id": "reviewer_demo_001",
    "comment": "删除绝对化和收益承诺表达后发布。",
    "revised_draft": "618 智能空调活动开启，限时下单享专属优惠，具体权益以活动页为准。",
}

high_final = review_graph.invoke(Command(resume=edit_decision), config=high_config)
print("publish_status:", high_final["publish_status"])
print("review_decision:", high_final["review_decision"])
print("final_content:", high_final["final_content"])
print("audit_log:")
for item in high_final["audit_log"]:
    print("-", item)


## 10. 对比输出：低风险自动通过，高风险可拒绝

同一张图会根据风险自动决定是否中断。低风险内容不会出现 `__interrupt__`；另一条高风险内容可以被人工拒绝。


In [ ]:
low_config = {"configurable": {"thread_id": "lg03-low-auto"}}
low_result = review_graph.invoke(
    {"request_id": "campaign_low_001"},
    config=low_config,
)
print("low_has_interrupt:", "__interrupt__" in low_result)
print("low_publish_status:", low_result["publish_status"])
print("low_review_decision:", low_result["review_decision"])
print("low_risk_score:", low_result["risk_score"])


In [ ]:
reject_config = {"configurable": {"thread_id": "lg03-high-reject"}}
reject_first = review_graph.invoke(
    {"request_id": "campaign_high_002"},
    config=reject_config,
)
print("reject_first_has_interrupt:", "__interrupt__" in reject_first)
print("reject_interrupt_terms:", [
    term
    for rule in reject_first["__interrupt__"][0].value["matched_rules"]
    for term in rule["terms"]
])

reject_decision = {
    "decision": "reject",
    "reviewer_id": "reviewer_demo_002",
    "comment": "内容包含医疗效果承诺，不能发布。",
}
reject_final = review_graph.invoke(Command(resume=reject_decision), config=reject_config)
print("reject_publish_status:", reject_final["publish_status"])
print("reject_final_content:", repr(reject_final["final_content"]))
print("reject_comment:", reject_final["reviewer_comment"])


## 11. 声明式中断：`interrupt_before`

`interrupt()` 适合在节点内部根据业务条件暂停。`interrupt_before` 适合拦截某个节点执行前的写操作，例如发布、扣款、发通知。

下面的图没有在节点里写 `interrupt()`，但会在 `publish_without_review` 执行前停住。


In [ ]:
def publish_without_review(state: ReviewState) -> dict[str, Any]:
    return {
        "publish_status": "published_without_review",
        "final_content": state["draft"],
        "audit_log": append_log(state, "声明式中断恢复后执行发布节点"),
    }


declarative_builder = StateGraph(ReviewState)
declarative_builder.add_node("load_request", load_request)
declarative_builder.add_node("auto_review", auto_review)
declarative_builder.add_node("publish_without_review", publish_without_review)
declarative_builder.add_edge(START, "load_request")
declarative_builder.add_edge("load_request", "auto_review")
declarative_builder.add_edge("auto_review", "publish_without_review")
declarative_builder.add_edge("publish_without_review", END)

declarative_graph = declarative_builder.compile(
    checkpointer=MemorySaver(),
    interrupt_before=["publish_without_review"],
)

declarative_config = {"configurable": {"thread_id": "lg03-interrupt-before"}}
declarative_first = declarative_graph.invoke(
    {"request_id": "campaign_high_001"},
    config=declarative_config,
)
declarative_snapshot = declarative_graph.get_state(declarative_config)
print("first keys:", sorted(declarative_first.keys()))
print("snapshot next:", declarative_snapshot.next)
print("publish_status exists before resume:", "publish_status" in declarative_first)

# 当前环境中，声明式中断恢复时传空 dict 即可继续执行下一个节点。
declarative_final = declarative_graph.invoke(Command(resume={}), config=declarative_config)
print("publish_status after resume:", declarative_final["publish_status"])
print("final_content_preview:", declarative_final["final_content"][:40])


## 12. 最终演示

最后一格把本节关键能力集中跑一遍：朴素实现会误发布，HITL 会暂停，高风险可以编辑后发布或拒绝，低风险会自动通过，声明式中断能拦截发布节点。


In [ ]:
print("=== 最终演示：朴素实现的问题 ===")
naive_final = naive_publish("campaign_high_001")
print("naive_publish_status:", naive_final["publish_status"])
print("naive_risk_score:", naive_final["risk_score"])
print("naive_matched_terms:", [term for rule in naive_final["matched_rules"] for term in rule["terms"]])

print("\n=== 最终演示：HITL 暂停与恢复 ===")
thread_id = f"lg03-final-edit-{uuid4()}"
config = {"configurable": {"thread_id": thread_id}}
first = review_graph.invoke({"request_id": "campaign_high_001"}, config=config)
snapshot = review_graph.get_state(config)
print("has_interrupt:", "__interrupt__" in first)
print("interrupt_action:", first["__interrupt__"][0].value["action"])
print("snapshot_next:", snapshot.next)
resumed = review_graph.invoke(Command(resume=edit_decision), config=config)
print("resumed_publish_status:", resumed["publish_status"])
print("resumed_final_content:", resumed["final_content"])

print("\n=== 最终演示：自动通过与拒绝 ===")
low_thread = {"configurable": {"thread_id": f"lg03-final-low-{uuid4()}"}}
low = review_graph.invoke({"request_id": "campaign_low_001"}, config=low_thread)
print("low_has_interrupt:", "__interrupt__" in low)
print("low_publish_status:", low["publish_status"])

reject_thread = {"configurable": {"thread_id": f"lg03-final-reject-{uuid4()}"}}
reject_start = review_graph.invoke({"request_id": "campaign_high_002"}, config=reject_thread)
reject_done = review_graph.invoke(Command(resume=reject_decision), config=reject_thread)
print("reject_has_interrupt:", "__interrupt__" in reject_start)
print("reject_publish_status:", reject_done["publish_status"])

print("\n=== 最终演示：声明式中断 ===")
declarative_thread = {"configurable": {"thread_id": f"lg03-final-before-{uuid4()}"}}
before_publish = declarative_graph.invoke({"request_id": "campaign_high_001"}, config=declarative_thread)
before_snapshot = declarative_graph.get_state(declarative_thread)
after_publish = declarative_graph.invoke(Command(resume={}), config=declarative_thread)
print("before_next:", before_snapshot.next)
print("before_has_publish_status:", "publish_status" in before_publish)
print("after_publish_status:", after_publish["publish_status"])


## 13. 练习

1. 增加 `needs_legal_review` 字段，让医疗效果相关文案进入法务审批。
2. 把 `edit_and_approve` 改成“编辑后重新进入自动审核”，确认修改后的文案风险下降。
3. 给 `audit_log` 增加 `reviewer_id` 和时间戳，形成完整审计记录。
4. 用 `interrupt_before` 拦截短信发送节点，比较它和节点内 `interrupt()` 的差异。
5. 增加一个条件边：低风险直接发布，高风险进入人工审批节点。

---
**下一节**: LG-04 持久化与记忆系统
